In [1]:
import pandas as pd

In [2]:
baseline_df = pd.read_csv("output/results_baselines.csv")
llama2_df = pd.read_csv("output/results_llama2-7b.csv")

In [3]:
clean_dict = {}

for i, row in llama2_df.iterrows():
    llama2 = row['Llama2 Affiliation']
    url = row["url"]
    timestamp = row["timestamp"]
    content = row["content"]
    desc = row["description"]
    headline = row["headline"]
    news = row['news name']

    curr_key = (news, timestamp, url, headline, desc, content)

    # Initialize the key in the dictionary if it doesn't exist
    if curr_key not in clean_dict:
        clean_dict[curr_key] = {"llama2": []}

    # Append the llama2 result
    clean_dict[curr_key]["llama2"].append(llama2)

In [30]:
baseline_df.columns

Index(['news', 'headline', 'description', 'content', 'timestamp', 'url',
       'NLTK Sentiment Scores', 'NLTK Sentiment (from compound)',
       'Top 10 most common words', 'Financial Sentiment',
       ...
       'Unnamed: 140', 'Unnamed: 141', 'Unnamed: 142', 'Unnamed: 143',
       'Unnamed: 144', 'Unnamed: 145', 'Unnamed: 146', 'Unnamed: 147',
       'Unnamed: 148', 'Unnamed: 149'],
      dtype='object', length=150)

In [4]:

rel_columns = ["NLTK Sentiment Scores", "NLTK Sentiment (from compound)", "Top 10 most common words", "Financial Sentiment", "News Sentiment", "Political Affiliation"]

for i, row in baseline_df.iterrows():
    # Extracting the metadata and the relevant columns
    url = row["url"]
    timestamp = row["timestamp"]
    content = row["content"]
    desc = row["description"]
    headline = row["headline"]
    news = row['news']

    curr_key = (news, timestamp, url, headline, desc, content)
    result = {}
    # Initialize the key in the dictionary if it doesn't exist
    if curr_key not in clean_dict:
        result = {rel: [row[rel]] for rel in rel_columns}
        clean_dict[curr_key] = result
    else:
        for rel in rel_columns:
            if rel in clean_dict[curr_key]:
                curr_rel = clean_dict[curr_key][rel]
                curr_rel.append(row[rel])
            else:
                result[rel] = [row[rel]]

    clean_dict[curr_key].update(result)

In [5]:
merged_df = pd.DataFrame.from_records(clean_dict)

TypeError: '<' not supported between instances of 'float' and 'str'

In [6]:
# A list to store all records
records = []

for curr_key, analyses in clean_dict.items():
    # curr_key contains your metadata as a tuple
    record = dict(zip(['news', 'timestamp', 'url', 'headline', 'desc', 'content'], curr_key))

    # Add analysis values to the record
    for rel_column, values in analyses.items():
        record[rel_column] = values

    records.append(record)

# Convert the list of records into a DataFrame
df = pd.DataFrame(records)


In [11]:
df.to_csv("output/results_baselines_llama2-7b.csv", sep=",", index=False)